# Crop Doctor – Notebook 02: Vision Model Training
**Hardware:** Google Colab Free GPU (T4) | **Target device:** Arm Cortex-A (Motorola Edge 60)

## What this notebook does
1. Mounts Google Drive and reads resized images (≤320 px).
2. Splits data: 75% train / 15% val / 10% holdout (stratified).
3. Fine-tunes **MobileNetV3-Large** on 25 crop-disease classes with strong augmentations.
4. Runs temperature calibration on the val set.
5. Evaluates on the held-out field photos.
6. Saves `best_mobilenet_v3_large.pth` and `metrics.json` to Drive.

In [ ]:
# ── CELL 1: Install dependencies & clone repo ────────────────────────────────
!pip install -q timm scikit-learn matplotlib pyyaml

from google.colab import drive
drive.mount('/content/drive')

!git clone --depth 1 https://github.com/varun-gajula/crop-doctor-offline.git /content/crop-doctor-offline 2>/dev/null || (cd /content/crop-doctor-offline && git pull)
import sys; sys.path.insert(0, '/content/crop-doctor-offline')
print('Setup complete.')

In [ ]:
# ── CELL 2: Config ───────────────────────────────────────────────────────────
import torch, os
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)

CFG = dict(
    data_dir   = '/content/drive/MyDrive/crop-doctor/data_raw',
    class_map  = '/content/crop-doctor-offline/data/class_map.yaml',
    output_dir = '/content/drive/MyDrive/crop-doctor/runs',
    model_name = 'mobilenet_v3_large',  # swap to 'convnext_tiny' to compare
    num_classes= 25,
    img_size   = 224,
    batch_size = 64,
    num_epochs = 20,
    lr         = 3e-4,
    label_smoothing = 0.1,
    seed       = 42,
)
os.makedirs(CFG['output_dir'], exist_ok=True)

In [ ]:
# ── CELL 3: Build dataset with stratified splits ─────────────────────────────
import random, numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from ml.dataset import CropDataset, get_transforms, scan_dataset

random.seed(CFG['seed']); np.random.seed(CFG['seed']); torch.manual_seed(CFG['seed'])

all_samples, class_meta = scan_dataset(CFG['data_dir'], CFG['class_map'])
print(f'Total images found: {len(all_samples)}')

# Separate real-field photos as dedicated holdout
field_samples  = [(p, l, f) for p, l, f in all_samples if f]
bench_samples  = [(p, l, f) for p, l, f in all_samples if not f]

labels_bench = [l for _, l, _ in bench_samples]
train_s, val_s = train_test_split(bench_samples, test_size=0.15, stratify=labels_bench, random_state=CFG['seed'])

print(f'Train: {len(train_s)}  Val: {len(val_s)}  Field Holdout: {len(field_samples)}')

train_tf = get_transforms(CFG['img_size'], is_train=True)
val_tf   = get_transforms(CFG['img_size'], is_train=False)

train_loader = DataLoader(CropDataset(train_s, train_tf), batch_size=CFG['batch_size'],
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(CropDataset(val_s,   val_tf),   batch_size=CFG['batch_size'],
                          shuffle=False, num_workers=2, pin_memory=True)
field_loader = DataLoader(CropDataset(field_samples, val_tf), batch_size=CFG['batch_size'],
                          shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
# ── CELL 4: Initialise model ─────────────────────────────────────────────────
from ml.model import create_model

model = create_model(CFG['model_name'], CFG['num_classes'], pretrained=True).to(DEVICE)
params_M = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Model: {CFG["model_name"]}  Parameters: {params_M:.2f}M')

In [ ]:
# ── CELL 5: Training loop with mixed-precision AMP ───────────────────────────
import torch.nn as nn
from torch.cuda.amp import GradScaler, autocast
from torch.optim.lr_scheduler import CosineAnnealingLR
import json

criterion  = nn.CrossEntropyLoss(label_smoothing=CFG['label_smoothing'])
optimizer  = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=1e-4)
scheduler  = CosineAnnealingLR(optimizer, T_max=CFG['num_epochs'])
scaler     = GradScaler(enabled=(DEVICE=='cuda'))

best_val_acc = 0.0
best_ckpt    = f"{CFG['output_dir']}/best_{CFG['model_name']}.pth"
log_rows     = []

for epoch in range(1, CFG['num_epochs']+1):
    # ---- Train ----
    model.train()
    train_correct = train_total = 0
    for imgs, labels, _ in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with autocast(enabled=(DEVICE=='cuda')):
            loss = criterion(model(imgs), labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        with torch.no_grad():
            preds = model(imgs).argmax(1)
            train_correct += (preds == labels).sum().item()
            train_total   += labels.size(0)

    # ---- Val ----
    model.eval()
    val_correct = val_total = 0
    with torch.no_grad():
        for imgs, labels, _ in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            preds = model(imgs).argmax(1)
            val_correct += (preds == labels).sum().item()
            val_total   += labels.size(0)

    train_acc = train_correct / train_total * 100
    val_acc   = val_correct   / val_total   * 100
    scheduler.step()

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_ckpt)

    row = dict(epoch=epoch, train_acc=round(train_acc,2), val_acc=round(val_acc,2), best_val_acc=round(best_val_acc,2))
    log_rows.append(row)
    print(f'Epoch {epoch:02d}/{CFG["num_epochs"]}  Train {train_acc:.1f}%  Val {val_acc:.1f}%  Best {best_val_acc:.1f}%')

print(f'\nTraining done. Best checkpoint saved to {best_ckpt}')
with open(f"{CFG['output_dir']}/train_log.json", 'w') as f:
    json.dump(log_rows, f, indent=2)

In [ ]:
# ── CELL 6: Temperature calibration + full evaluation ────────────────────────
from ml.model import TemperatureScaler
from ml.evaluate import evaluate_model

model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))

scaler_T = TemperatureScaler()
T = scaler_T.fit(val_loader, model, DEVICE)
print(f'Calibrated temperature T = {T:.4f}')

# Save temperature
import json
with open(f"{CFG['output_dir']}/temperature.json", 'w') as f:
    json.dump({'temperature': T, 'model': CFG['model_name']}, f)

print('\n=== Validation Set Evaluation ===')
val_metrics = evaluate_model(model, val_loader, class_meta, DEVICE, scaler_T, CFG['output_dir'])
print(f'Val Top-1: {val_metrics["overall_top1_accuracy"]}%')

if len(field_samples) > 0:
    print('\n=== Real Field Holdout Evaluation (Domain Gap Audit) ===')
    field_metrics = evaluate_model(model, field_loader, class_meta, DEVICE, scaler_T,
                                    f"{CFG['output_dir']}/field_holdout")
    print(f'Field Top-1: {field_metrics["overall_top1_accuracy"]}%')

print('\nConfidence Threshold Sweep:')
for row in val_metrics['confidence_threshold_sweep']:
    print(f'  T={row["threshold"]}  Coverage={row["coverage_pct"]}%  Accuracy={row["accuracy_pct"]}%')

In [ ]:
# ── CELL 7: Save final model package ─────────────────────────────────────────
import shutil, json

# Save to Drive
pkg_dir = f"{CFG['output_dir']}/model_package"
os.makedirs(pkg_dir, exist_ok=True)

shutil.copy(best_ckpt, f"{pkg_dir}/vision_weights.pth")

# Generate labels.json (index -> class_id string for Android)
import yaml
with open(CFG['class_map'], 'r', encoding='utf-8') as f:
    class_data = yaml.safe_load(f)['classes']
labels = {str(k): v['class_id'] for k, v in class_data.items()}
with open(f"{pkg_dir}/labels.json", 'w', encoding='utf-8') as f:
    json.dump(labels, f, ensure_ascii=False, indent=2)

# Generate preprocess.json (Android must replicate exact normalisation)
preprocess = {
    'input_size': 224,
    'mean': [0.485, 0.456, 0.406],
    'std': [0.229, 0.224, 0.225],
    'channels': 'RGB'
}
with open(f"{pkg_dir}/preprocess.json", 'w') as f:
    json.dump(preprocess, f, indent=2)

print(f'Model package saved to {pkg_dir}')
print('Next step: Run notebook 03_export_vision.ipynb to convert to ExecuTorch .pte')